<a href="https://colab.research.google.com/github/addadugurudurga2024-lang/Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I build the feature vector using five historical performance features:
GSC impressions, GSC clicks, GSC average position, GA4 sessions, and GA4
engaged sessions.

Missing numeric values are filled with 0 for count-based features. For
average position, missing values are kept as missing and handled separately
because missing search data does not necessarily mean zero position.

No categorical feature is used in the final vector because client and
content identifiers are context fields rather than meaningful predictive
categories.

In [6]:
feature_query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
LIMIT 1000
"""

X = con.sql(feature_query).df()

count_features = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X[count_features] = X[count_features].fillna(0)

print("Feature vector shape:", X.shape)
display(X.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (1000, 5)


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,20,0,3.350000,0,0
1,1,0,0.000000,0,0
2,125,1,4.928000,0,0
3,7,0,4.000000,0,0
4,11,0,2.272727,0,0


## 2. Feature notes

| Feature | Meaning | Missing-value handling | Available when? |
|---|---|---|---|
| `gsc_impressions` | Number of search impressions observed for the content. | Missing values are filled with 0. | Available from historical search data before the decision moment. |
| `gsc_clicks` | Number of search clicks observed for the content. | Missing values are filled with 0. | Available from historical search data before the decision moment. |
| `gsc_avg_position` | Average search position observed for the content. | Missing values remain missing because missing search data does not mean position 0. | Available from historical search data before the decision moment. |
| `ga4_sessions` | Number of website sessions observed for the content. | Missing values are filled with 0. | Available from historical analytics data before the decision moment. |
| `ga4_engaged_sessions` | Number of engaged sessions observed for the content. | Missing values are filled with 0. | Available from historical analytics data before the decision moment. |

No categorical variables are included in the final feature vector.

In [7]:
# Check missing values after the feature preparation

print("Missing values after filling:")
display(X.isna().sum())

Missing values after filling:


,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,238
ga4_sessions,0
ga4_engaged_sessions,0


## 3. The leakage hunt

I check the feature set for three common leakage risks.

First, I check whether any feature is directly derived from the label.
Second, I check whether any feature uses a future reporting period.
Third, I check whether identifier or product-status fields have accidentally
entered the feature vector.

The final feature vector should contain only historical information that
would be available before the prediction decision.

In [8]:
# Leakage checks

feature_columns = set(X.columns)

label_derived = [
    "Needs_Refresh",
    "refresh_label",
    "future_outcome"
]

future_features = [
    "future_clicks",
    "future_impressions",
    "future_sessions"
]

excluded_identifiers = [
    "client_hash_id",
    "content_hash_id"
]

print("Label-derived columns present:",
      feature_columns.intersection(label_derived))

print("Future-derived columns present:",
      feature_columns.intersection(future_features))

print("Identifier columns present:",
      feature_columns.intersection(excluded_identifiers))

Label-derived columns present: set()
Future-derived columns present: set()
Identifier columns present: set()


### Leakage result

No label-derived, future-window, or identifier fields are present in the
final feature vector.

The feature vector therefore contains only the five historical performance
signals selected for the refresh-review decision.

This is a leakage check rather than proof of model performance.

## 4. What I excluded and why

- `client_hash_id` — excluded because it identifies the client and is used
  only for grouping/context, not as a meaningful predictive feature.

- `content_hash_id` — excluded because it identifies the content item and
  does not represent a meaningful content-performance signal.

- `report_date` — excluded from the feature vector because it is the time
  context rather than a direct performance feature.

- `gsc_data_available` — excluded because it describes data availability,
  not content performance.

- `ga4_data_available` — excluded because it describes data availability,
  not content performance.

- Future-period performance fields — excluded because they would not be
  available at the prediction moment and could cause data leakage.

- `Needs_Refresh` — excluded from the feature vector because it is the
  target/proxy rather than an input feature.

In [9]:
# Final feature list

print("Final features:")
for feature in X.columns:
    print("-", feature)

print("\nNumber of features:", len(X.columns))

Final features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

Number of features: 5


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.